# 08 · Food Relief Priority Index

Combine P(High ∪ Severe) with a food-fragility score. Fragility scalers are fit on TRAIN only, then applied to the test set. Maps rendered with folium for Ida (LA) and Ian (FL).

In [3]:
import sys, os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 60)
from config import (
    DATA_PATHS, HURRICANE_META, STATES_IN_SCOPE,
    TARGET_COL, TARGET_CLASS_COL, FEATURE_GROUPS,
    RANDOM_STATE, SEVERITY_BINS, SEVERITY_LABELS,
)
RAW = DATA_PATHS['raw']; INTERIM = DATA_PATHS['interim']
PROC = DATA_PATHS['processed']; MODELS = DATA_PATHS['models']
OUT = DATA_PATHS['outputs']


In [4]:
import joblib, folium
from src.priority_index import fit_fragility_scalers, apply_fragility, priority_index, save_scalers
df = pd.read_csv(PROC / 'abt_with_clusters.csv', dtype={'zip_code': str}).dropna(subset=[TARGET_CLASS_COL])
pkg = joblib.load(MODELS / 'best_classifier.pkl')
pipe, le, features = pkg['pipe'], pkg['label_encoder'], pkg['features']

In [5]:
# Fit fragility scalers on TRAIN only
train = df[df['train_test_split']=='TRAIN']
scalers = fit_fragility_scalers(train)
save_scalers(scalers, MODELS / 'fragility_scalers.pkl')
df = apply_fragility(df, scalers)

In [6]:
# Predict probabilities for TEST set
te = df[df['train_test_split']=='TEST'].copy()
proba = pipe.predict_proba(te[features])
te_prio = priority_index(te, proba, list(le.classes_))
top = te_prio.sort_values('priority_rank').head(20)
top[['priority_rank','zip_code','state','hurricane_name',
     'food_relief_priority_index','prob_high_or_severe',
     'food_fragility_score','svi_overall','damage_severity_class']]

c:\Users\chaitanya\Documents\ML Project\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,priority_rank,zip_code,state,hurricane_name,food_relief_priority_index,prob_high_or_severe,food_fragility_score,svi_overall,damage_severity_class
668468,1,33438,FL,Ian,0.903375,1.000000,0.758438,0.979488,High
703680,2,33476,FL,Ian,0.899456,0.999999,0.748642,0.998744,Medium
703681,2,33476,FL,Ian,0.899456,0.999999,0.748642,0.998744,Medium
703682,2,33476,FL,Ian,0.899456,0.999999,0.748642,0.998744,Medium
703683,2,33476,FL,Ian,0.899456,0.999999,0.748642,0.998744,Medium
703652,2,33476,FL,Ian,0.899456,0.999999,0.748642,0.998744,Low
703658,2,33476,FL,Ian,0.899456,0.999999,0.748642,0.998744,Low
703653,2,33476,FL,Ian,0.899456,0.999999,0.748642,0.998744,Low
703661,2,33476,FL,Ian,0.899456,0.999999,0.748642,0.998744,Low
703668,2,33476,FL,Ian,0.899456,0.999999,0.748642,0.998744,Low


In [7]:
te_prio.to_csv(OUT / 'priority_rankings.csv', index=False)
print('saved priority_rankings.csv', te_prio.shape)

saved priority_rankings.csv (1372353, 71)


## Folium choropleth maps — Ida and Ian

In [9]:
import geopandas as gpd
zcta_path = next((RAW / 'zcta').rglob('*.shp'))
zcta = gpd.read_file(zcta_path).to_crs('EPSG:4326')
zcta['zip_code'] = zcta['ZCTA5CE20'].astype(str).str.zfill(5)

TOP_N = 200  # render only highest-priority ZIPs to keep map tractable

for hn, center in [('Ida', (30.2, -90.9)), ('Ian', (26.6, -81.9))]:
    # Dedupe: take the max priority per zip within this hurricane
    sub = (te_prio[te_prio['hurricane_name'] == hn]
           .groupby('zip_code', as_index=False)
           .agg(priority_index_norm=('priority_index_norm', 'max'),
                priority_rank=('priority_rank', 'min'),
                food_relief_priority_index=('food_relief_priority_index', 'max'),
                prob_high_or_severe=('prob_high_or_severe', 'max'),
                food_fragility_score=('food_fragility_score', 'max'))
           .nsmallest(TOP_N, 'priority_rank'))

    z = zcta.merge(sub, on='zip_code')
    if z.empty:
        print('no zips for', hn); continue

    # Simplify polygon geometry to shrink embedded GeoJSON ~10x
    z['geometry'] = z.geometry.simplify(tolerance=0.001, preserve_topology=True)

    m = folium.Map(location=center, zoom_start=7, tiles='cartodbpositron')

    folium.Choropleth(
        geo_data=z, data=z,
        columns=['zip_code', 'priority_index_norm'],
        key_on='feature.properties.zip_code',
        fill_color='YlOrRd', fill_opacity=0.7, line_opacity=0.2,
        legend_name=f'Priority index — {hn} (top {TOP_N})',
    ).add_to(m)

    # ONE transparent GeoJson layer for tooltips, instead of per-row layers
    folium.GeoJson(
        z, name='details',
        style_function=lambda _: {'fillOpacity': 0, 'color': 'transparent'},
        tooltip=folium.GeoJsonTooltip(
            fields=['zip_code', 'priority_rank', 'priority_index_norm',
                    'prob_high_or_severe', 'food_fragility_score'],
            aliases=['ZIP', 'Rank', 'Priority (norm)', 'P(High∪Severe)', 'Fragility'],
            localize=True),
    ).add_to(m)

    out = OUT / f'priority_map_{hn.lower()}.html'
    m.save(str(out)); print(f'saved {out}  ({len(z)} ZIPs)')

saved C:\Users\chaitanya\Documents\ML Project\outputs\priority_map_ida.html  (200 ZIPs)
saved C:\Users\chaitanya\Documents\ML Project\outputs\priority_map_ian.html  (200 ZIPs)


## Top-50 vs bottom-50 summary

In [10]:
top50 = te_prio.nsmallest(50, 'priority_rank')
bot50 = te_prio.nlargest(50, 'priority_rank')
pd.DataFrame({
  'top50': [top50['food_desert_flag'].sum(), top50['svi_overall'].mean(),
            top50[TARGET_COL].mean()],
  'bot50': [bot50['food_desert_flag'].sum(), bot50['svi_overall'].mean(),
            bot50[TARGET_COL].mean()],
}, index=['food_desert_count','mean_svi','mean_damage_per_1k'])

,top50,bot50
food_desert_count,50.000000,0.000000
mean_svi,0.998359,0.191617
mean_damage_per_1k,6.548186,3.286308
